In [5]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\chank\AppData\Local\Python\pythoncore-3.14-64\python.exe
3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]


In [1]:
import sys
print(sys.executable)

c:\Users\chank\OneDrive\Desktop\ML\venv\Scripts\python.exe


In [2]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [3]:
import sys

print(sys.executable)

c:\Users\chank\OneDrive\Desktop\ML\venv\Scripts\python.exe


In [4]:
import tensorflow as tf

print(tf.__version__)

2.21.0


In [5]:
import os

print("Train folders:")
print(os.listdir("archive/train"))

print("\nValidation folders:")
print(os.listdir("archive/val"))

Train folders:


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'archive/train'

In [6]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles and folders here:")
print(os.listdir())

Current folder:
c:\Users\chank\OneDrive\Desktop\ML

Files and folders here:
['archive', 'biodeg.ipynb', 'venv']


In [7]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    "archive/Dataset/train",
    image_size=(224,224),
    batch_size=32,
    label_mode="binary"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "archive/Dataset/val",
    image_size=(224,224),
    batch_size=32,
    label_mode="binary"
)

Found 5812 files belonging to 2 classes.
Found 1201 files belonging to 2 classes.


In [8]:
import os

for root, dirs, files in os.walk("archive"):
    print(root)

archive
archive\Dataset
archive\Dataset\train
archive\Dataset\train\biodegradable
archive\Dataset\train\non_biodegradable
archive\Dataset\val
archive\Dataset\val\biodegradable
archive\Dataset\val\non_biodegradable


In [13]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

In [14]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
 30/182 ━━━━━━━━━━━━━━━━━━━━ 1:05 434ms/step - accuracy: 0.8891 - loss: 0.3012

InvalidArgumentError: Graph execution error:

Detected at node decode_image/DecodeImage defined at (most recent call last):
<stack traces unavailable>
jpeg::Uncompress failed. Invalid JPEG data or crop window.
	 [[{{node decode_image/DecodeImage}}]]
	 [[IteratorGetNext]] [Op:__inference_multi_step_on_iterator_9004]

In [15]:
import os
from PIL import Image

bad_files = []

for root, dirs, files in os.walk("archive"):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            path = os.path.join(root, file)
            try:
                img = Image.open(path)
                img.verify()
            except Exception as e:
                bad_files.append(path)
                print("Corrupt:", path)

print("\nTotal corrupt files:", len(bad_files))

ModuleNotFoundError: No module named 'PIL'

In [9]:
from tensorflow.keras import layers
import tensorflow as tf

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))

x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [10]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
 26/182 ━━━━━━━━━━━━━━━━━━━━ 1:16 487ms/step - accuracy: 0.6128 - loss: 0.6666

InvalidArgumentError: Graph execution error:

Detected at node decode_image/DecodeImage defined at (most recent call last):
<stack traces unavailable>
jpeg::Uncompress failed. Invalid JPEG data or crop window.
	 [[{{node decode_image/DecodeImage}}]]
	 [[IteratorGetNext]] [Op:__inference_multi_step_on_iterator_9004]

In [11]:
import tensorflow as tf

for images, labels in train_ds.take(1):
    print("Images shape:", images.shape)
    print("Labels shape:", labels.shape)
    print("Labels dtype:", labels.dtype)
    

Images shape: (32, 224, 224, 3)
Labels shape: (32, 1)
Labels dtype: <dtype: 'float32'>


In [12]:
print(train_ds.class_names)

['biodegradable', 'non_biodegradable']


In [6]:
import sys

print("Python:", sys.executable)
print("Version:", sys.version)

Python: c:\Users\chank\AppData\Local\Python\pythoncore-3.14-64\python.exe
Version: 3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]


In [7]:
import sys

print(sys.executable)

c:\Users\chank\AppData\Local\Python\pythoncore-3.14-64\python.exe


In [8]:
import tensorflow as tf

print(tf.__version__)

ModuleNotFoundError: No module named 'tensorflow'